# 📉 Customer Churn Prediction — Notebook Training (Google Colab)

Demonstrasi **data mining** end-to-end: dari data mentah → pembersihan → pemodelan → penemuan pola tersembunyi → artefak siap-deploy.

**Cara pakai:** jalankan sel dari atas ke bawah (`Shift+Enter`). Di sel terakhir, model akan otomatis terunduh untuk dipakai di web app Streamlit.

| Tahap | Isi |
|---|---|
| 0 | Setup & import |
| 1 | Ambil dataset |
| 2 | EDA singkat |
| 3 | Pembersihan data |
| 4 | Split train/test |
| 5 | Pipeline preprocessing |
| 6 | Latih 2 model |
| 7 | Evaluasi & pilih pemenang |
| 8 | Feature importance (pola tersembunyi) |
| 9 | Simpan & unduh artefak |


## Langkah 0 — Setup & import
Colab sudah punya pandas & scikit-learn; kita cukup pastikan versinya cukup baru lalu import semua yang dibutuhkan.

In [ ]:
!pip install -q --upgrade scikit-learn joblib

import json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report,
)

RANDOM_STATE = 42
TARGET = "Churn"
print("Setup selesai.")

## Langkah 1 — Ambil dataset
Dataset **Telco Customer Churn** (IBM) ditarik langsung dari URL publik — tanpa upload manual. Berisi 7.043 pelanggan telekomunikasi dengan 21 kolom.

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)
print("Dimensi:", df.shape)          # (7043, 21)
df.head()

## Langkah 2 — EDA singkat
Kita cek tiga hal yang menentukan keputusan desain berikutnya:
1. **Distribusi target** — seimbang atau tidak?
2. **Tipe data** — ada yang aneh?
3. **Kualitas** — nilai hilang/tersembunyi?

In [ ]:
# 1. Distribusi target
print("Distribusi Churn:")
print(df["Churn"].value_counts(normalize=True).round(3))

# 2. Tipe data — perhatikan TotalCharges
print("\nTipe TotalCharges:", df["TotalCharges"].dtype, "(harusnya numerik!)")

# 3. Kualitas: TotalCharges tersimpan sebagai teks; berapa yang kosong?
kosong = pd.to_numeric(df["TotalCharges"], errors="coerce").isna().sum()
print("Baris TotalCharges kosong:", kosong, "(pelanggan baru, tenure 0)")

> **Temuan EDA:** target **tidak seimbang** (~26,5% churn) → nanti pakai `class_weight="balanced"` & jangan andalkan accuracy. `TotalCharges` ternyata **teks dengan 11 nilai kosong** → perlu dibersihkan. Ini contoh nyata "data mentah" yang akan kamu ceritakan di laporan.

## Langkah 3 — Pembersihan data
Tiga aksi: buang identifier, perbaiki `TotalCharges`, jadikan target biner.

In [ ]:
df = df.drop(columns=["customerID"])                       # identifier, bukan fitur
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")  # teks -> angka (kosong jadi NaN)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})       # 0/1 -> kategori
df[TARGET] = (df[TARGET] == "Yes").astype(int)            # target -> 1=churn

X = df.drop(columns=[TARGET])
y = df[TARGET]
print("Fitur:", X.shape[1], "| Target churn rate:", round(y.mean(), 3))

## Langkah 4 — Split train/test (sebelum preprocessing!)
Membagi data **dulu**, baru preprocessing, agar scaler/encoder tidak "mengintip" data test (*data leakage*). `stratify=y` menjaga proporsi churn sama di kedua set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])

## Langkah 5 — Pipeline preprocessing
`ColumnTransformer` memproses numerik & kategorikal secara terpisah, lalu dibungkus jadi satu objek. Karena terbungkus `Pipeline`, semua transformasi **hanya belajar dari data train**.

In [ ]:
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])
print("Numerik:", num_cols)
print("Kategorikal:", len(cat_cols), "kolom")

## Langkah 6 — Latih dua model
**Logistic Regression** (baseline sederhana) vs **Gradient Boosting** (model utama). Keduanya pakai `class_weight="balanced"` untuk mengatasi data tidak seimbang.

In [ ]:
candidates = {
    "Logistic Regression (baseline)": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Gradient Boosting": HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=400, max_depth=4,
        l2_regularization=1.0, class_weight="balanced",
        random_state=RANDOM_STATE),
}

def evaluate(model, name):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {"model": name,
            "accuracy": round(accuracy_score(y_test, pred), 4),
            "precision": round(precision_score(y_test, pred), 4),
            "recall": round(recall_score(y_test, pred), 4),
            "f1": round(f1_score(y_test, pred), 4),
            "roc_auc": round(roc_auc_score(y_test, proba), 4)}

results, fitted = [], {}
for name, clf in candidates.items():
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    results.append(evaluate(pipe, name))

pd.DataFrame(results).set_index("model")

## Langkah 7 — Evaluasi & pilih pemenang
Datanya tidak seimbang, jadi **jangan pilih berdasarkan accuracy**. Kita pilih berdasarkan **F1** (menyeimbangkan menangkap churner vs alarm palsu) — keputusan yang perlu kamu **justifikasi di laporan**.

In [ ]:
leaderboard = pd.DataFrame(results).sort_values("f1", ascending=False)
best_name = leaderboard.iloc[0]["model"]
best_model = fitted[best_name]
print("Model terpilih:", best_name, "\n")

proba = best_model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print(classification_report(y_test, pred, target_names=["Bertahan", "Churn"]))

cm = confusion_matrix(y_test, pred)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm)

## Langkah 8 — Feature importance (pola tersembunyi)
`permutation_importance`: seberapa besar ROC-AUC turun saat sebuah fitur diacak. Inilah "pengetahuan yang diekstrak" dari data — driver churn yang sesungguhnya.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=10,
    random_state=RANDOM_STATE, scoring="roc_auc")

importance = (pd.DataFrame({"feature": X_test.columns,
                            "importance": perm.importances_mean})
              .sort_values("importance", ascending=False)
              .reset_index(drop=True))
importance.head(10)

In [ ]:
# Visualisasi cepat
import matplotlib.pyplot as plt
top = importance.head(10).iloc[::-1]
plt.figure(figsize=(7, 4.5))
plt.barh(top["feature"], top["importance"], color="#2563eb")
plt.xlabel("Penurunan ROC-AUC saat fitur diacak")
plt.title("Top 10 Faktor Pendorong Churn")
plt.tight_layout()
plt.show()

## Langkah 9 — Simpan & unduh artefak
**Langkah krusial.** Session Colab akan hilang, jadi model & metadata harus diunduh. Web app Streamlit hanya butuh **dua file** ini — semua grafik di app digambar dari `metadata.json`.

In [ ]:
# Kumpulkan semua yang dibutuhkan app ke dalam satu metadata
fpr, tpr, _ = roc_curve(y_test, proba)
idx = np.linspace(0, len(fpr) - 1, min(60, len(fpr))).astype(int)  # ringkas ke ~60 titik

metadata = {
    "best_model": best_name,
    "leaderboard": results,
    "n_rows": int(len(df)),
    "churn_rate": round(float(y.mean()), 4),
    "confusion_matrix": cm.tolist(),
    "roc": {"fpr": [round(float(fpr[i]), 4) for i in idx],
            "tpr": [round(float(tpr[i]), 4) for i in idx],
            "auc": round(float(roc_auc_score(y_test, proba)), 4)},
    "top_features": importance.head(10).to_dict(orient="records"),
    "schema": {
        "numeric": {c: {"min": float(X[c].min()), "max": float(X[c].max()),
                        "median": float(X[c].median())} for c in num_cols},
        "categorical": {c: sorted(X[c].dropna().unique().tolist()) for c in cat_cols},
        "feature_order": X.columns.tolist(),
    },
}

joblib.dump(best_model, "churn_model.joblib")
with open("metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Tersimpan: churn_model.joblib + metadata.json")

In [ ]:
# Unduh ke komputermu (khusus Colab)
from google.colab import files
files.download("churn_model.joblib")
files.download("metadata.json")

---
### ✅ Selesai
Kamu sekarang punya `churn_model.joblib` + `metadata.json`. Lanjut ke **Bagian B**: taruh dua file ini bersama `app.py` & `requirements.txt` di repo GitHub, lalu deploy gratis ke [Streamlit Community Cloud](https://share.streamlit.io). Petunjuk lengkap ada di `PANDUAN_DEPLOY.md`.